# NYC Motor Vehicle Collisions
## Milestone 1: Data Exploration, Cleaning, Integration and Visualization

### Course
Data Engineering and Visualization – Winter Semester 2025

### Project Objective
This project performs a complete data engineering workflow using real-world
New York City motor vehicle collision data.

The project includes:

1. Loading the NYC Crashes and Person datasets.
2. Exploring the structure and quality of both datasets.
3. Performing pre-integration data cleaning.
4. Aggregating person-level records.
5. Integrating the datasets using `COLLISION_ID`.
6. Performing post-integration cleaning.
7. Validating the final integrated dataset.
8. Creating visualizations that answer meaningful research questions.
9. Exporting a dashboard-ready dataset.


> **Execution note**  
> This saved-output copy was executed with a deterministic offline demonstration sample because the execution sandbox could not connect to the NYC Open Data download endpoint. The workflow, schemas, cleaning, integration, validation, charts, and exports are fully executed. For final submission results, use the original notebook and set `USE_SAMPLE = False` in Google Colab or Jupyter with internet access.


## Dataset Description

### Primary Dataset: Motor Vehicle Collisions – Crashes

Each row represents one reported collision and contains information including:

- Collision date and time
- Borough and ZIP code
- Latitude and longitude
- Number of people injured or killed
- Contributing factors
- Vehicle types
- Unique collision identifier

### Secondary Dataset: Motor Vehicle Collisions – Person

Each row represents one person involved in a collision and includes:

- Person type
- Injury status
- Age
- Sex
- Pedestrian or cyclist information
- Vehicle association
- Collision identifier

### Integration Key

The datasets are integrated using:

`COLLISION_ID`

Because several people can be involved in the same collision, the Person dataset
must first be aggregated to one row per collision before integration.

In [45]:
import importlib

required_packages = ["pandas", "numpy", "matplotlib", "plotly", "pyarrow"]
for package in required_packages:
    module = importlib.import_module(package)
    print(f"{package}: available ({getattr(module, '__version__', 'version unavailable')})")

pandas: available (2.2.2)
numpy: available (2.0.2)
matplotlib: available (3.10.0)
plotly: available (5.24.1)
pyarrow: available (18.1.0)


In [46]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

RANDOM_STATE = 42
START_YEAR = 2012
END_YEAR = 2025

## Research Questions

Each team member is responsible for at least two research questions.

### Member 1

1. How did collision frequency and casualty rates change between 2012 and 2025?
2. Which boroughs experienced the greatest increase or decrease in collision severity?

### Member 2

3. How do collision patterns vary by hour, weekday and weekend?
4. Are nighttime collisions associated with more injuries or deaths?

### Member 3

5. Which contributing factors have the highest injury and fatality rates?
6. Which combinations of contributing factor and vehicle type are most dangerous?

### Member 4

7. How do pedestrian and cyclist involvement rates vary across boroughs?
8. Which age groups experience the highest injury severity for each person type?

### Member 5

9. Where are the persistent geographic collision hotspots in New York City?
10. Are multi-person collisions more severe than collisions involving fewer people?

## Team Contributions

Replace the placeholder names with the actual team members.

| Team Member | Research Questions | Cleaning Contribution | Website Contribution |
|---|---|---|---|
| Member 1 | Questions 1 and 2 | Date, time and numeric cleaning | KPI cards and annual trends |
| Member 2 | Questions 3 and 4 | Borough and location cleaning | Time heatmap and filters |
| Member 3 | Questions 5 and 6 | Contributing factor and vehicle cleaning | Factor and vehicle charts |
| Member 4 | Questions 7 and 8 | Person dataset cleaning and aggregation | Person-type visualizations |
| Member 5 | Questions 9 and 10 | Integration and post-integration validation | Map, search mode and deployment |

## 1. Loading the Datasets

The datasets are loaded directly from NYC Open Data.

The full datasets are large. During development, `USE_SAMPLE` can be set to
`True`. Before producing final results, it should be set to `False`.

In [47]:

# -----------------------------------------------------------------------------
# EXECUTED DEMONSTRATION MODE
# -----------------------------------------------------------------------------
# The source notebook loads the official datasets directly from NYC Open Data.
# This executed copy uses a deterministic offline sample with the same important
# columns and data-quality issues so the full workflow can run in this sandbox.

rng = np.random.default_rng(RANDOM_STATE)

n_crashes = 5_000
base_collision_ids = np.arange(5_000_000, 5_000_000 + n_crashes)

boroughs = np.array(["BROOKLYN", "QUEENS", "MANHATTAN", "BRONX", "STATEN ISLAND", None], dtype=object)
borough_probs = [0.31, 0.27, 0.20, 0.17, 0.04, 0.01]

factors = np.array([
    "Driver Inattention/Distraction",
    "Failure to Yield Right-of-Way",
    "Following Too Closely",
    "Unsafe Speed",
    "Traffic Control Disregarded",
    "Alcohol Involvement",
    "Backing Unsafely",
    "Unspecified",
    None
], dtype=object)
factor_probs = [0.25, 0.13, 0.12, 0.08, 0.07, 0.03, 0.07, 0.22, 0.03]

vehicles = np.array([
    "Sedan", "Station Wagon/Sport Utility Vehicle", "Taxi", "Bike",
    "Bus", "Motorcycle", "Pick-up Truck", "Van", "Unknown", None
], dtype=object)
vehicle_probs = [0.35, 0.25, 0.09, 0.07, 0.05, 0.04, 0.05, 0.04, 0.04, 0.02]

start = np.datetime64("2012-07-01")
end = np.datetime64("2025-12-31")
day_offsets = rng.integers(0, int((end - start).astype("timedelta64[D]").astype(int)) + 1, size=n_crashes)
crash_dates = start + day_offsets.astype("timedelta64[D]")
hours = rng.choice(np.arange(24), size=n_crashes, p=np.array([
    0.025,0.018,0.015,0.013,0.014,0.022,0.035,0.055,0.070,0.052,0.045,0.043,
    0.045,0.047,0.050,0.058,0.070,0.082,0.075,0.055,0.045,0.038,0.032,0.026
]) / np.array([
    0.025,0.018,0.015,0.013,0.014,0.022,0.035,0.055,0.070,0.052,0.045,0.043,
    0.045,0.047,0.050,0.058,0.070,0.082,0.075,0.055,0.045,0.038,0.032,0.026
]).sum())
minutes = rng.integers(0, 60, size=n_crashes)
crash_times = [f"{h:02d}:{m:02d}" for h, m in zip(hours, minutes)]

borough_values = rng.choice(boroughs, size=n_crashes, p=borough_probs)
borough_centers = {
    "BROOKLYN": (40.6501, -73.9496),
    "QUEENS": (40.7282, -73.7949),
    "MANHATTAN": (40.7831, -73.9712),
    "BRONX": (40.8448, -73.8648),
    "STATEN ISLAND": (40.5795, -74.1502),
    None: (40.7128, -74.0060),
}
latitudes = np.array([borough_centers[b][0] for b in borough_values]) + rng.normal(0, 0.035, n_crashes)
longitudes = np.array([borough_centers[b][1] for b in borough_values]) + rng.normal(0, 0.045, n_crashes)

# Inject invalid geographic values for domain-rule cleaning.
invalid_geo_idx = rng.choice(n_crashes, size=35, replace=False)
latitudes[invalid_geo_idx[:18]] = 0
longitudes[invalid_geo_idx[18:]] = 0

injured = rng.poisson(0.32, n_crashes)
killed = rng.binomial(1, 0.006, n_crashes)
# Rare but plausible extreme cases.
injured[rng.choice(n_crashes, size=8, replace=False)] += rng.integers(4, 10, 8)

ped_injured = np.minimum(injured, rng.binomial(injured, 0.22))
cyclist_injured = np.minimum(injured - ped_injured, rng.binomial(np.maximum(injured - ped_injured, 0), 0.15))
motorist_injured = np.maximum(injured - ped_injured - cyclist_injured, 0)
ped_killed = killed * rng.binomial(1, 0.45, n_crashes)
cyclist_killed = (killed - ped_killed) * rng.binomial(1, 0.20, n_crashes)
motorist_killed = np.maximum(killed - ped_killed - cyclist_killed, 0)

zip_map = {
    "BROOKLYN": "11201", "QUEENS": "11368", "MANHATTAN": "10001",
    "BRONX": "10451", "STATEN ISLAND": "10301", None: None
}
zip_codes = [zip_map[b] for b in borough_values]

street_names = np.array(["BROADWAY", "ATLANTIC AVENUE", "QUEENS BOULEVARD", "GRAND CONCOURSE", "HYLAN BOULEVARD", None], dtype=object)

crash_dict = {
    "CRASH DATE": pd.to_datetime(crash_dates).strftime("%m/%d/%Y"),
    "CRASH TIME": crash_times,
    "BOROUGH": borough_values,
    "ZIP CODE": zip_codes,
    "LATITUDE": latitudes,
    "LONGITUDE": longitudes,
    "LOCATION": [f"POINT ({lon:.6f} {lat:.6f})" if lat != 0 and lon != 0 else None for lat, lon in zip(latitudes, longitudes)],
    "ON STREET NAME": rng.choice(street_names, n_crashes),
    "CROSS STREET NAME": rng.choice(street_names, n_crashes),
    "OFF STREET NAME": rng.choice(street_names, n_crashes),
    "NUMBER OF PERSONS INJURED": injured,
    "NUMBER OF PERSONS KILLED": killed,
    "NUMBER OF PEDESTRIANS INJURED": ped_injured,
    "NUMBER OF PEDESTRIANS KILLED": ped_killed,
    "NUMBER OF CYCLIST INJURED": cyclist_injured,
    "NUMBER OF CYCLIST KILLED": cyclist_killed,
    "NUMBER OF MOTORIST INJURED": motorist_injured,
    "NUMBER OF MOTORIST KILLED": motorist_killed,
    "CONTRIBUTING FACTOR VEHICLE 1": rng.choice(factors, n_crashes, p=factor_probs),
    "CONTRIBUTING FACTOR VEHICLE 2": rng.choice(factors, n_crashes, p=factor_probs),
    "CONTRIBUTING FACTOR VEHICLE 3": rng.choice(factors, n_crashes, p=factor_probs),
    "CONTRIBUTING FACTOR VEHICLE 4": rng.choice(factors, n_crashes, p=factor_probs),
    "CONTRIBUTING FACTOR VEHICLE 5": rng.choice(factors, n_crashes, p=factor_probs),
    "COLLISION_ID": base_collision_ids,
    "VEHICLE TYPE CODE 1": rng.choice(vehicles, n_crashes, p=vehicle_probs),
    "VEHICLE TYPE CODE 2": rng.choice(vehicles, n_crashes, p=vehicle_probs),
    "VEHICLE TYPE CODE 3": rng.choice(vehicles, n_crashes, p=vehicle_probs),
    "VEHICLE TYPE CODE 4": rng.choice(vehicles, n_crashes, p=vehicle_probs),
    "VEHICLE TYPE CODE 5": rng.choice(vehicles, n_crashes, p=vehicle_probs),
}

df_crashes_raw = pd.DataFrame(crash_dict)

# Add duplicate and invalid-key rows to demonstrate cleaning.
df_crashes_raw = pd.concat([df_crashes_raw, df_crashes_raw.iloc[:10]], ignore_index=True)
invalid_crash_row = df_crashes_raw.iloc[[0]].copy()
invalid_crash_row["COLLISION_ID"] = np.nan
invalid_crash_row["NUMBER OF PERSONS INJURED"] = -2
df_crashes_raw = pd.concat([df_crashes_raw, invalid_crash_row], ignore_index=True)

# Generate person rows for most collisions from April 2016 onward.
crash_dates_pd = pd.to_datetime(crash_dict["CRASH DATE"])
eligible_mask = crash_dates_pd >= pd.Timestamp("2016-04-01")
eligible_ids = base_collision_ids[eligible_mask]
eligible_dates = crash_dates_pd[eligible_mask]
eligible_times = np.array(crash_times, dtype=object)[eligible_mask]

keep_mask = rng.random(len(eligible_ids)) < 0.88
matched_ids = eligible_ids[keep_mask]
matched_dates = eligible_dates[keep_mask]
matched_times = eligible_times[keep_mask]

person_rows = []
unique_id = 20_000_000
person_types = ["Occupant", "Pedestrian", "Bicyclist", "Other"]
person_type_probs = [0.70, 0.16, 0.10, 0.04]
sex_values = ["M", "F", "U", None]
sex_probs = [0.50, 0.43, 0.05, 0.02]

for collision_id, cdate, ctime in zip(matched_ids, matched_dates, matched_times):
    n_people = int(np.clip(rng.poisson(1.8) + 1, 1, 7))
    for p_idx in range(n_people):
        ptype = rng.choice(person_types, p=person_type_probs)
        injury_draw = rng.random()
        injury = "Killed" if injury_draw < 0.004 else ("Injured" if injury_draw < 0.18 else "Unspecified")
        age = int(np.clip(rng.normal(38, 19), 0, 95))
        person_rows.append({
            "UNIQUE_ID": unique_id,
            "COLLISION_ID": int(collision_id),
            "CRASH_DATE": cdate.strftime("%m/%d/%Y"),
            "CRASH_TIME": ctime,
            "PERSON_ID": f"P{unique_id}",
            "PERSON_TYPE": ptype,
            "PERSON_INJURY": injury,
            "VEHICLE_ID": f"V{collision_id}-{max(1, p_idx)}" if ptype == "Occupant" else None,
            "PERSON_AGE": age,
            "EJECTION": rng.choice(["Not Ejected", "Ejected", "Unknown", None], p=[0.87,0.02,0.08,0.03]),
            "EMOTIONAL_STATUS": rng.choice(["Conscious", "Unknown", None], p=[0.73,0.20,0.07]),
            "BODILY_INJURY": rng.choice(["Entire Body", "Head", "Back", "Unknown", None], p=[0.10,0.12,0.10,0.55,0.13]),
            "POSITION_IN_VEHICLE": "Driver" if ptype == "Occupant" and p_idx == 0 else ("Passenger" if ptype == "Occupant" else None),
            "SAFETY_EQUIPMENT": rng.choice(["Lap Belt & Harness", "Helmet", "Unknown", None], p=[0.45,0.08,0.35,0.12]),
            "PED_LOCATION": rng.choice(["At Intersection", "Not at Intersection", None], p=[0.18,0.08,0.74]) if ptype in ["Pedestrian", "Bicyclist"] else None,
            "PED_ACTION": rng.choice(["Crossing", "Riding", "Unknown", None], p=[0.15,0.10,0.10,0.65]) if ptype in ["Pedestrian", "Bicyclist"] else None,
            "COMPLAINT": rng.choice(["Pain", "None Visible", "Unknown", None], p=[0.16,0.55,0.20,0.09]),
            "PED_ROLE": ptype if ptype in ["Pedestrian", "Bicyclist"] else None,
            "CONTRIBUTING_FACTOR_1": rng.choice(["Unspecified", "Failure to Yield", "Unsafe Speed", None], p=[0.65,0.12,0.08,0.15]),
            "CONTRIBUTING_FACTOR_2": rng.choice(["Unspecified", "Unknown", None], p=[0.55,0.20,0.25]),
            "PERSON_SEX": rng.choice(sex_values, p=sex_probs),
        })
        unique_id += 1

# Add a few malformed ages and duplicates.
df_persons_raw = pd.DataFrame(person_rows)
if len(df_persons_raw) >= 10:
    df_persons_raw.loc[df_persons_raw.index[:3], "PERSON_AGE"] = [-5, 150, 999]
    df_persons_raw = pd.concat([df_persons_raw, df_persons_raw.iloc[:20]], ignore_index=True)
    invalid_person = df_persons_raw.iloc[[0]].copy()
    invalid_person["COLLISION_ID"] = np.nan
    invalid_person["UNIQUE_ID"] = unique_id
    df_persons_raw = pd.concat([df_persons_raw, invalid_person], ignore_index=True)

USE_SAMPLE = True
crash_row_limit = len(df_crashes_raw)
person_row_limit = len(df_persons_raw)

print("Offline demonstration datasets generated successfully.")
print(f"Crash rows: {len(df_crashes_raw):,}")
print(f"Person rows: {len(df_persons_raw):,}")


Offline demonstration datasets generated successfully.
Crash rows: 5,011
Person rows: 8,977


In [48]:
dataset_dimensions = pd.DataFrame({
    "Dataset": ["Crashes", "Persons"],
    "Rows": [
        df_crashes_raw.shape[0],
        df_persons_raw.shape[0]
    ],
    "Columns": [
        df_crashes_raw.shape[1],
        df_persons_raw.shape[1]
    ]
})

display(dataset_dimensions)

,Dataset,Rows,Columns
0,Crashes,5011,29
1,Persons,8977,21


In [49]:
print("Crashes dataset preview:")
display(df_crashes_raw.head())

print("\nPersons dataset preview:")
display(df_persons_raw.head())

Crashes dataset preview:


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,09/14/2013,01:04,MANHATTAN,10001,40.83,-73.91,POINT (-73.913096 40.833095),GRAND CONCOURSE,None,GRAND CONCOURSE,0,0,0,0,0,0,0,0,Unspecified,Unsafe Speed,Backing Unsafely,Driver Inattention/Distraction,Driver Inattention/Distraction,"5,000,000.00",Station Wagon/Sport Utility Vehicle,Pick-up Truck,Station Wagon/Sport Utility Vehicle,Sedan,Sedan
1,12/13/2022,17:30,MANHATTAN,10001,40.84,-73.92,POINT (-73.923245 40.844848),ATLANTIC AVENUE,BROADWAY,QUEENS BOULEVARD,2,0,0,0,1,0,1,0,Driver Inattention/Distraction,Unsafe Speed,Unspecified,Driver Inattention/Distraction,Failure to Yield Right-of-Way,"5,000,001.00",Motorcycle,Sedan,Sedan,Station Wagon/Sport Utility Vehicle,Sedan
2,05/03/2021,22:34,BROOKLYN,11201,40.66,-73.92,POINT (-73.920122 40.661667),BROADWAY,None,QUEENS BOULEVARD,1,0,0,0,0,0,1,0,Driver Inattention/Distraction,Traffic Control Disregarded,Unsafe Speed,Driver Inattention/Distraction,Backing Unsafely,"5,000,002.00",Bus,Sedan,Motorcycle,Sedan,Bus
3,06/04/2018,19:43,BROOKLYN,11201,40.61,-73.90,POINT (-73.901439 40.607136),ATLANTIC AVENUE,HYLAN BOULEVARD,BROADWAY,0,0,0,0,0,0,0,0,Failure to Yield Right-of-Way,Driver Inattention/Distraction,Failure to Yield Right-of-Way,Following Too Closely,Unspecified,"5,000,003.00",Taxi,Taxi,Sedan,Motorcycle,Pick-up Truck
4,05/06/2018,09:34,BROOKLYN,11201,40.71,-73.95,POINT (-73.947107 40.712138),BROADWAY,BROADWAY,None,0,0,0,0,0,0,0,0,Driver Inattention/Distraction,Unspecified,Driver Inattention/Distraction,Traffic Control Disregarded,Following Too Closely,"5,000,004.00",Sedan,Unknown,Motorcycle,Sedan,Sedan



Persons dataset preview:


,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX
0,20000000,"5,000,001.00",12/13/2022,17:30,P20000000,Pedestrian,Unspecified,None,-5,Not Ejected,Conscious,Unknown,None,Unknown,Not at Intersection,None,None Visible,Pedestrian,Unspecified,Unspecified,F
1,20000001,"5,000,001.00",12/13/2022,17:30,P20000001,Occupant,Unspecified,V5000001-1,150,Not Ejected,None,Unknown,Passenger,Helmet,None,None,None Visible,None,Unspecified,None,F
2,20000002,"5,000,001.00",12/13/2022,17:30,P20000002,Occupant,Unspecified,V5000001-2,999,Not Ejected,Conscious,Unknown,Passenger,Unknown,None,None,Pain,None,Unspecified,Unspecified,None
3,20000003,"5,000,001.00",12/13/2022,17:30,P20000003,Occupant,Unspecified,V5000001-3,11,Not Ejected,Conscious,Unknown,Passenger,Lap Belt & Harness,None,None,None Visible,None,Unspecified,Unspecified,M
4,20000004,"5,000,002.00",05/03/2021,22:34,P20000004,Occupant,Unspecified,V5000002-1,28,Not Ejected,Conscious,Unknown,Driver,Unknown,None,None,Unknown,None,Unspecified,None,F


In [50]:
def convert_to_snake_case(column_name):
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    return column_name.strip("_")


df_crashes = df_crashes_raw.copy()
df_persons = df_persons_raw.copy()

df_crashes.columns = [
    convert_to_snake_case(column)
    for column in df_crashes.columns
]

df_persons.columns = [
    convert_to_snake_case(column)
    for column in df_persons.columns
]

print("Crashes columns:")
print(df_crashes.columns.tolist())

print("\nPerson columns:")
print(df_persons.columns.tolist())

Crashes columns:
['crash_date', 'crash_time', 'borough', 'zip_code', 'latitude', 'longitude', 'location', 'on_street_name', 'cross_street_name', 'off_street_name', 'number_of_persons_injured', 'number_of_persons_killed', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed', 'number_of_cyclist_injured', 'number_of_cyclist_killed', 'number_of_motorist_injured', 'number_of_motorist_killed', 'contributing_factor_vehicle_1', 'contributing_factor_vehicle_2', 'contributing_factor_vehicle_3', 'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5', 'collision_id', 'vehicle_type_code_1', 'vehicle_type_code_2', 'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5']

Person columns:
['unique_id', 'collision_id', 'crash_date', 'crash_time', 'person_id', 'person_type', 'person_injury', 'vehicle_id', 'person_age', 'ejection', 'emotional_status', 'bodily_injury', 'position_in_vehicle', 'safety_equipment', 'ped_location', 'ped_action', 'complaint', 'ped_role', 'contri

In [51]:
print("CRASHES DATASET INFORMATION")
print("-" * 60)
df_crashes.info()

print("\nPERSON DATASET INFORMATION")
print("-" * 60)
df_persons.info()

CRASHES DATASET INFORMATION
------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5011 entries, 0 to 5010
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   crash_date                     5011 non-null   object 
 1   crash_time                     5011 non-null   object 
 2   borough                        4952 non-null   object 
 3   zip_code                       4952 non-null   object 
 4   latitude                       5011 non-null   float64
 5   longitude                      5011 non-null   float64
 6   location                       4976 non-null   object 
 7   on_street_name                 4166 non-null   object 
 8   cross_street_name              4206 non-null   object 
 9   off_street_name                4130 non-null   object 
 10  number_of_persons_injured      5011 non-null   int64  
 11  number_of_persons_k

In [52]:
def create_missing_value_report(dataframe):
    report = pd.DataFrame({
        "missing_count": dataframe.isna().sum(),
        "missing_percentage": (
            dataframe.isna().mean() * 100
        ).round(2),
        "data_type": dataframe.dtypes.astype(str)
    })

    return report.sort_values(
        by="missing_percentage",
        ascending=False
    )


crashes_missing_before = create_missing_value_report(df_crashes)
persons_missing_before = create_missing_value_report(df_persons)

print("Crashes missing-value report:")
display(crashes_missing_before)

print("\nPersons missing-value report:")
display(persons_missing_before)

Crashes missing-value report:


,missing_count,missing_percentage,data_type
off_street_name,881,17.58,object
on_street_name,845,16.86,object
cross_street_name,805,16.06,object
contributing_factor_vehicle_2,175,3.49,object
contributing_factor_vehicle_1,155,3.09,object
contributing_factor_vehicle_4,153,3.05,object
contributing_factor_vehicle_5,142,2.83,object
contributing_factor_vehicle_3,139,2.77,object
vehicle_type_code_4,112,2.24,object
vehicle_type_code_5,103,2.06,object



Persons missing-value report:


,missing_count,missing_percentage,data_type
ped_location,8370,93.24,object
ped_action,8109,90.33,object
ped_role,6581,73.31,object
vehicle_id,2778,30.95,object
position_in_vehicle,2778,30.95,object
contributing_factor_2,2187,24.36,object
contributing_factor_1,1366,15.22,object
bodily_injury,1205,13.42,object
safety_equipment,1091,12.15,object
complaint,805,8.97,object


In [53]:
crashes_missing_plot = (
    crashes_missing_before
    .query("missing_percentage > 0")
    .reset_index()
    .rename(columns={"index": "column"})
    .head(20)
)

fig = px.bar(
    crashes_missing_plot,
    x="missing_percentage",
    y="column",
    orientation="h",
    title="Top Missing-Value Percentages in Crashes Dataset",
    labels={
        "missing_percentage": "Missing values (%)",
        "column": "Column"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [54]:
duplicate_summary = pd.DataFrame({
    "Metric": [
        "Exact duplicate crash rows",
        "Exact duplicate person rows",
        "Missing crash COLLISION_ID values",
        "Missing person COLLISION_ID values",
        "Duplicated crash COLLISION_ID values"
    ],
    "Value": [
        df_crashes.duplicated().sum(),
        df_persons.duplicated().sum(),
        df_crashes["collision_id"].isna().sum(),
        df_persons["collision_id"].isna().sum(),
        df_crashes["collision_id"].duplicated().sum()
    ]
})

display(duplicate_summary)

,Metric,Value
0,Exact duplicate crash rows,10
1,Exact duplicate person rows,20
2,Missing crash COLLISION_ID values,1
3,Missing person COLLISION_ID values,1
4,Duplicated crash COLLISION_ID values,10


## 2. Initial Exploratory Data Analysis

The initial EDA is performed before cleaning to discover:

- Incorrect data types
- Missing values
- Duplicate records
- Invalid ages
- Invalid coordinates
- Rare or inconsistent categories
- Unusual injury and fatality values
- Temporal and geographic patterns

In [55]:
print("Crashes numerical statistics:")
display(df_crashes.describe(include=[np.number]).T)

print("\nPersons numerical statistics:")
display(df_persons.describe(include=[np.number]).T)

Crashes numerical statistics:


,count,mean,std,min,25%,50%,75%,max
latitude,"5,011.00",40.58,2.44,0.00,40.67,40.73,40.79,40.95
longitude,"5,011.00",-73.65,4.30,-74.28,-73.97,-73.91,-73.83,0.00
number_of_persons_injured,"5,011.00",0.33,0.63,-2.00,0.00,0.00,1.00,10.00
number_of_persons_killed,"5,011.00",0.01,0.08,0.00,0.00,0.00,0.00,1.00
number_of_pedestrians_injured,"5,011.00",0.08,0.28,0.00,0.00,0.00,0.00,3.00
number_of_pedestrians_killed,"5,011.00",0.00,0.05,0.00,0.00,0.00,0.00,1.00
number_of_cyclist_injured,"5,011.00",0.04,0.21,0.00,0.00,0.00,0.00,3.00
number_of_cyclist_killed,"5,011.00",0.00,0.02,0.00,0.00,0.00,0.00,1.00
number_of_motorist_injured,"5,011.00",0.21,0.48,0.00,0.00,0.00,0.00,7.00
number_of_motorist_killed,"5,011.00",0.00,0.05,0.00,0.00,0.00,0.00,1.00



Persons numerical statistics:


,count,mean,std,min,25%,50%,75%,max
unique_id,"8,977.00","20,004,468.04","2,591.50","20,000,000.00","20,002,224.00","20,004,468.00","20,006,712.00","20,008,956.00"
collision_id,"8,976.00","5,002,471.43","1,449.02","5,000,001.00","5,001,216.75","5,002,453.00","5,003,737.00","5,004,999.00"
person_age,"8,977.00",38.08,23.55,-5.00,25.00,38.00,51.00,999.00


In [56]:
crash_category_columns = [
    "borough",
    "contributing_factor_vehicle_1",
    "vehicle_type_code_1"
]

for column in crash_category_columns:
    if column in df_crashes.columns:
        print(f"\nTop values for {column}:")
        display(
            df_crashes[column]
            .value_counts(dropna=False)
            .head(15)
            .to_frame("count")
        )


Top values for borough:


,count
borough,
BROOKLYN,1545
QUEENS,1335
MANHATTAN,1052
BRONX,840
STATEN ISLAND,180
None,59



Top values for contributing_factor_vehicle_1:


,count
contributing_factor_vehicle_1,
Driver Inattention/Distraction,1199
Unspecified,1147
Failure to Yield Right-of-Way,686
Following Too Closely,599
Unsafe Speed,405
Backing Unsafely,344
Traffic Control Disregarded,326
None,155
Alcohol Involvement,150



Top values for vehicle_type_code_1:


,count
vehicle_type_code_1,
Sedan,1736
Station Wagon/Sport Utility Vehicle,1288
Taxi,475
Bike,348
Bus,257
Unknown,230
Pick-up Truck,222
Van,195
Motorcycle,165


In [57]:
person_category_columns = [
    "person_type",
    "person_injury",
    "person_sex",
    "ejection"
]

for column in person_category_columns:
    if column in df_persons.columns:
        print(f"\nTop values for {column}:")
        display(
            df_persons[column]
            .value_counts(dropna=False)
            .head(15)
            .to_frame("count")
        )


Top values for person_type:


,count
person_type,
Occupant,6199
Pedestrian,1498
Bicyclist,898
Other,382



Top values for person_injury:


,count
person_injury,
Unspecified,7296
Injured,1645
Killed,36



Top values for person_sex:


,count
person_sex,
M,4530
F,3826
U,429
None,192



Top values for ejection:


,count
ejection,
Not Ejected,7889
Unknown,651
None,260
Ejected,177


In [58]:
if "borough" in df_crashes.columns:
    borough_counts_before = (
        df_crashes["borough"]
        .value_counts(dropna=False)
        .rename_axis("borough")
        .reset_index(name="crash_count")
    )

    fig = px.bar(
        borough_counts_before,
        x="borough",
        y="crash_count",
        title="Initial Number of Collisions by Borough",
        labels={
            "borough": "Borough",
            "crash_count": "Number of collisions"
        }
    )

    fig.show()

## 3. Pre-Integration Cleaning

### Crashes Dataset Cleaning Decisions

- Records without a valid `COLLISION_ID` are removed because they cannot be
  reliably identified or integrated.
- Duplicate collision IDs are removed because the crash table should contain
  one row per collision.
- Dates and times are converted to proper datetime values.
- Numeric injury and fatality columns are converted to numeric types.
- Negative casualty counts are considered impossible and replaced with missing
  values before being filled with zero.
- Missing boroughs are represented as `UNKNOWN` instead of guessing a borough.
- Invalid geographic coordinates are replaced with missing values.
- Missing contributing factors and vehicle types are represented as `Unknown`.
- Rare but valid high casualty values are retained rather than automatically
  removed using IQR.

In [59]:
def standardize_text(series, missing_label="Unknown"):
    cleaned_series = (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    cleaned_series = cleaned_series.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })

    return cleaned_series.fillna(missing_label)


def standardize_title_case(series, missing_label="Unknown"):
    return standardize_text(
        series,
        missing_label=missing_label
    ).str.title()

In [60]:
crashes_clean = df_crashes.copy()

crashes_clean["collision_id"] = pd.to_numeric(
    crashes_clean["collision_id"],
    errors="coerce"
).astype("Int64")

crashes_clean = crashes_clean.dropna(subset=["collision_id"])
crashes_clean = crashes_clean.drop_duplicates()
crashes_clean = crashes_clean.drop_duplicates(
    subset=["collision_id"],
    keep="first"
)

crashes_clean["crash_date"] = pd.to_datetime(
    crashes_clean["crash_date"],
    errors="coerce"
)

crashes_clean["crash_time"] = (
    crashes_clean["crash_time"]
    .astype("string")
    .str.strip()
)

crashes_clean["crash_datetime"] = pd.to_datetime(
    crashes_clean["crash_date"].dt.strftime("%Y-%m-%d")
    + " "
    + crashes_clean["crash_time"],
    errors="coerce"
)

crashes_clean["year"] = crashes_clean["crash_datetime"].dt.year
crashes_clean["month"] = crashes_clean["crash_datetime"].dt.month
crashes_clean["month_name"] = crashes_clean["crash_datetime"].dt.month_name()
crashes_clean["weekday"] = crashes_clean["crash_datetime"].dt.day_name()
crashes_clean["hour"] = crashes_clean["crash_datetime"].dt.hour

crashes_clean = crashes_clean[
    crashes_clean["year"].between(
        START_YEAR,
        END_YEAR,
        inclusive="both"
    )
].copy()

crashes_clean["is_weekend"] = (
    crashes_clean["weekday"]
    .isin(["Saturday", "Sunday"])
)

crashes_clean["time_period"] = pd.cut(
    crashes_clean["hour"],
    bins=[-1, 5, 11, 17, 21, 23],
    labels=[
        "Late Night",
        "Morning",
        "Afternoon",
        "Evening",
        "Night"
    ]
)

print("Crash date and time cleaning completed.")

Crash date and time cleaning completed.


In [61]:
crashes_clean["borough"] = (
    standardize_text(
        crashes_clean["borough"],
        missing_label="UNKNOWN"
    )
    .str.upper()
)

crashes_clean["zip_code"] = (
    crashes_clean["zip_code"]
    .astype("string")
    .str.extract(r"(\d{5})", expand=False)
)

street_columns = [
    "on_street_name",
    "cross_street_name",
    "off_street_name"
]

for column in street_columns:
    if column in crashes_clean.columns:
        crashes_clean[column] = (
            standardize_text(
                crashes_clean[column],
                missing_label="Unknown"
            )
            .str.upper()
        )

factor_columns = [
    column for column in crashes_clean.columns
    if column.startswith("contributing_factor_vehicle")
]

for column in factor_columns:
    crashes_clean[column] = standardize_title_case(
        crashes_clean[column]
    )

vehicle_columns = [
    column for column in crashes_clean.columns
    if column.startswith("vehicle_type_code")
]

for column in vehicle_columns:
    crashes_clean[column] = standardize_title_case(
        crashes_clean[column]
    )

casualty_columns = [
    "number_of_persons_injured",
    "number_of_persons_killed",
    "number_of_pedestrians_injured",
    "number_of_pedestrians_killed",
    "number_of_cyclist_injured",
    "number_of_cyclist_killed",
    "number_of_motorist_injured",
    "number_of_motorist_killed"
]

for column in casualty_columns:
    if column in crashes_clean.columns:
        numeric_values = pd.to_numeric(
            crashes_clean[column],
            errors="coerce"
        )

        numeric_values = numeric_values.mask(numeric_values < 0)

        crashes_clean[column] = (
            numeric_values
            .fillna(0)
            .round()
            .astype("int64")
        )

crashes_clean["latitude"] = pd.to_numeric(
    crashes_clean["latitude"],
    errors="coerce"
)

crashes_clean["longitude"] = pd.to_numeric(
    crashes_clean["longitude"],
    errors="coerce"
)

valid_coordinates = (
    crashes_clean["latitude"].between(40.40, 41.00)
    & crashes_clean["longitude"].between(-74.30, -73.60)
)

crashes_clean.loc[
    ~valid_coordinates,
    ["latitude", "longitude"]
] = np.nan

print("Crash category, casualty and location cleaning completed.")

Crash category, casualty and location cleaning completed.


In [62]:
crashes_clean["total_casualties"] = (
    crashes_clean["number_of_persons_injured"]
    + crashes_clean["number_of_persons_killed"]
)

crashes_clean["has_injury"] = (
    crashes_clean["number_of_persons_injured"] > 0
)

crashes_clean["has_fatality"] = (
    crashes_clean["number_of_persons_killed"] > 0
)

crashes_clean["severity_category"] = np.select(
    [
        crashes_clean["number_of_persons_killed"] > 0,
        crashes_clean["number_of_persons_injured"] >= 2,
        crashes_clean["number_of_persons_injured"] == 1
    ],
    [
        "Fatal Collision",
        "Multiple Injuries",
        "Single Injury"
    ],
    default="No Reported Casualty"
)

crashes_clean["primary_contributing_factor"] = (
    crashes_clean["contributing_factor_vehicle_1"]
)

crashes_clean["primary_vehicle_type"] = (
    crashes_clean["vehicle_type_code_1"]
)

display(
    crashes_clean[
        [
            "collision_id",
            "crash_datetime",
            "borough",
            "year",
            "hour",
            "time_period",
            "total_casualties",
            "severity_category"
        ]
    ].head()
)

,collision_id,crash_datetime,borough,year,hour,time_period,total_casualties,severity_category
0,5000000,2013-09-14 01:04:00,MANHATTAN,2013,1,Late Night,0,No Reported Casualty
1,5000001,2022-12-13 17:30:00,MANHATTAN,2022,17,Afternoon,2,Multiple Injuries
2,5000002,2021-05-03 22:34:00,BROOKLYN,2021,22,Night,1,Single Injury
3,5000003,2018-06-04 19:43:00,BROOKLYN,2018,19,Evening,0,No Reported Casualty
4,5000004,2018-05-06 09:34:00,BROOKLYN,2018,9,Morning,0,No Reported Casualty


In [63]:
def calculate_iqr_report(dataframe, columns):
    results = []

    for column in columns:
        if column not in dataframe.columns:
            continue

        first_quartile = dataframe[column].quantile(0.25)
        third_quartile = dataframe[column].quantile(0.75)
        iqr = third_quartile - first_quartile

        lower_bound = first_quartile - (1.5 * iqr)
        upper_bound = third_quartile + (1.5 * iqr)

        potential_outliers = (
            (dataframe[column] < lower_bound)
            | (dataframe[column] > upper_bound)
        ).sum()

        results.append({
            "column": column,
            "q1": first_quartile,
            "q3": third_quartile,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "potential_outliers": potential_outliers
        })

    return pd.DataFrame(results)


crash_outlier_report = calculate_iqr_report(
    crashes_clean,
    [
        "number_of_persons_injured",
        "number_of_persons_killed",
        "total_casualties"
    ]
)

display(crash_outlier_report)

,column,q1,q3,iqr,lower_bound,upper_bound,potential_outliers
0,number_of_persons_injured,0.00,1.00,1.00,-1.50,2.50,31
1,number_of_persons_killed,0.00,0.00,0.00,0.00,0.00,29
2,total_casualties,0.00,1.00,1.00,-1.50,2.50,31


### Crash Outlier Decision

The IQR method identifies unusually high casualty values. These records are not
automatically deleted because a collision involving several injured people may
be rare but still valid.

Only values that violate clear domain rules, such as negative injury counts or
coordinates outside the selected geographic range, are treated as invalid.

### Person Dataset Cleaning Decisions

- Person records without a valid collision ID are removed because they cannot
  be integrated.
- Exact duplicates and duplicated person identifiers are removed.
- Person ages below 0 or above 110 are replaced with missing values.
- Missing age values are not replaced with the global mean because age varies
  significantly between person types.
- Person types, injury statuses and sex values are standardized.
- The Person dataset remains person-level until it is aggregated by collision.

In [64]:
persons_clean = df_persons.copy()

persons_clean["collision_id"] = pd.to_numeric(
    persons_clean["collision_id"],
    errors="coerce"
).astype("Int64")

if "unique_id" in persons_clean.columns:
    persons_clean["unique_id"] = pd.to_numeric(
        persons_clean["unique_id"],
        errors="coerce"
    ).astype("Int64")

persons_clean = persons_clean.dropna(subset=["collision_id"])
persons_clean = persons_clean.drop_duplicates()

if "unique_id" in persons_clean.columns:
    persons_clean = persons_clean.drop_duplicates(
        subset=["unique_id"],
        keep="first"
    )
else:
    persons_clean = persons_clean.drop_duplicates(
        subset=["collision_id", "person_id"],
        keep="first"
    )

persons_clean["crash_date"] = pd.to_datetime(
    persons_clean["crash_date"],
    errors="coerce"
)

persons_clean["person_year"] = persons_clean["crash_date"].dt.year

persons_clean = persons_clean[
    persons_clean["person_year"].between(
        START_YEAR,
        END_YEAR,
        inclusive="both"
    )
].copy()

persons_clean["person_age"] = pd.to_numeric(
    persons_clean["person_age"],
    errors="coerce"
)

persons_clean.loc[
    ~persons_clean["person_age"].between(0, 110),
    "person_age"
] = np.nan

print("Basic person cleaning completed.")

Basic person cleaning completed.


In [65]:
raw_person_type = standardize_text(
    persons_clean["person_type"],
    missing_label="Unknown"
).str.upper()

persons_clean["person_type_clean"] = np.select(
    [
        raw_person_type.str.contains(
            "PEDESTRIAN",
            na=False
        ),
        raw_person_type.str.contains(
            "BICYCLIST|CYCLIST",
            regex=True,
            na=False
        ),
        raw_person_type.str.contains(
            "OCCUPANT",
            na=False
        )
    ],
    [
        "Pedestrian",
        "Cyclist",
        "Motor Vehicle Occupant"
    ],
    default="Other or Unknown"
)

raw_injury = standardize_text(
    persons_clean["person_injury"],
    missing_label="Unspecified"
).str.upper()

persons_clean["person_injury_clean"] = np.select(
    [
        raw_injury.str.contains("KILLED", na=False),
        raw_injury.str.contains("INJURED", na=False)
    ],
    [
        "Killed",
        "Injured"
    ],
    default="Unspecified"
)

raw_sex = standardize_text(
    persons_clean["person_sex"],
    missing_label="Unknown"
).str.upper()

persons_clean["person_sex_clean"] = raw_sex.replace({
    "M": "Male",
    "MALE": "Male",
    "F": "Female",
    "FEMALE": "Female",
    "U": "Unknown"
})

persons_clean.loc[
    ~persons_clean["person_sex_clean"].isin(
        ["Male", "Female"]
    ),
    "person_sex_clean"
] = "Unknown"

display(
    persons_clean[
        [
            "collision_id",
            "person_id",
            "person_type_clean",
            "person_injury_clean",
            "person_age",
            "person_sex_clean"
        ]
    ].head()
)

,collision_id,person_id,person_type_clean,person_injury_clean,person_age,person_sex_clean
0,5000001,P20000000,Pedestrian,Unspecified,NaN,Female
1,5000001,P20000001,Motor Vehicle Occupant,Unspecified,NaN,Female
2,5000001,P20000002,Motor Vehicle Occupant,Unspecified,NaN,Unknown
3,5000001,P20000003,Motor Vehicle Occupant,Unspecified,11.00,Male
4,5000002,P20000004,Motor Vehicle Occupant,Unspecified,28.00,Female


In [66]:
pre_integration_quality = pd.DataFrame({
    "Metric": [
        "Crash rows",
        "Crash exact duplicates",
        "Crash duplicated collision IDs",
        "Crash missing collision IDs",
        "Person rows",
        "Person exact duplicates",
        "Person missing collision IDs",
        "Invalid person ages"
    ],
    "Before Cleaning": [
        len(df_crashes),
        df_crashes.duplicated().sum(),
        df_crashes["collision_id"].duplicated().sum(),
        df_crashes["collision_id"].isna().sum(),
        len(df_persons),
        df_persons.duplicated().sum(),
        df_persons["collision_id"].isna().sum(),
        pd.to_numeric(
            df_persons["person_age"],
            errors="coerce"
        ).pipe(
            lambda values: (
                (values < 0) | (values > 110)
            ).sum()
        )
    ],
    "After Cleaning": [
        len(crashes_clean),
        crashes_clean.duplicated().sum(),
        crashes_clean["collision_id"].duplicated().sum(),
        crashes_clean["collision_id"].isna().sum(),
        len(persons_clean),
        persons_clean.duplicated().sum(),
        persons_clean["collision_id"].isna().sum(),
        (
            (persons_clean["person_age"] < 0)
            | (persons_clean["person_age"] > 110)
        ).sum()
    ]
})

display(pre_integration_quality)

,Metric,Before Cleaning,After Cleaning
0,Crash rows,5011,5000
1,Crash exact duplicates,10,0
2,Crash duplicated collision IDs,10,0
3,Crash missing collision IDs,1,0
4,Person rows,8977,8956
5,Person exact duplicates,20,0
6,Person missing collision IDs,1,0
7,Invalid person ages,7,0


## 4. Person-Level Aggregation

A direct merge between Crashes and Person would create multiple rows for the
same collision because one collision can involve several people.

To preserve one row per collision, the following person-level information is
aggregated:

- Number of person records
- Pedestrian count
- Cyclist count
- Motor vehicle occupant count
- Number of injured people
- Number of killed people
- Male and female counts
- Average and median age

In [67]:
persons_clean["is_pedestrian"] = (
    persons_clean["person_type_clean"] == "Pedestrian"
).astype("int8")

persons_clean["is_cyclist"] = (
    persons_clean["person_type_clean"] == "Cyclist"
).astype("int8")

persons_clean["is_occupant"] = (
    persons_clean["person_type_clean"]
    == "Motor Vehicle Occupant"
).astype("int8")

persons_clean["is_injured_person"] = (
    persons_clean["person_injury_clean"] == "Injured"
).astype("int8")

persons_clean["is_killed_person"] = (
    persons_clean["person_injury_clean"] == "Killed"
).astype("int8")

persons_clean["is_male"] = (
    persons_clean["person_sex_clean"] == "Male"
).astype("int8")

persons_clean["is_female"] = (
    persons_clean["person_sex_clean"] == "Female"
).astype("int8")

In [68]:
person_summary = (
    persons_clean
    .groupby("collision_id", as_index=False)
    .agg(
        person_records=("collision_id", "size"),
        unique_persons=("person_id", "nunique"),
        pedestrians_involved=("is_pedestrian", "sum"),
        cyclists_involved=("is_cyclist", "sum"),
        occupants_involved=("is_occupant", "sum"),
        injured_person_records=("is_injured_person", "sum"),
        killed_person_records=("is_killed_person", "sum"),
        male_persons=("is_male", "sum"),
        female_persons=("is_female", "sum"),
        average_person_age=("person_age", "mean"),
        median_person_age=("person_age", "median"),
        minimum_person_age=("person_age", "min"),
        maximum_person_age=("person_age", "max")
    )
)

age_summary_columns = [
    "average_person_age",
    "median_person_age",
    "minimum_person_age",
    "maximum_person_age"
]

person_summary[age_summary_columns] = (
    person_summary[age_summary_columns].round(1)
)

print("Person summary dimensions:", person_summary.shape)
display(person_summary.head())

Person summary dimensions: (3191, 14)


,collision_id,person_records,unique_persons,pedestrians_involved,cyclists_involved,occupants_involved,injured_person_records,killed_person_records,male_persons,female_persons,average_person_age,median_person_age,minimum_person_age,maximum_person_age
0,5000001,4,4,1,0,3,0,0,1,2,11.00,11.00,11.00,11.00
1,5000002,4,4,0,0,4,1,0,1,3,41.80,41.50,28.00,56.00
2,5000003,4,4,0,1,3,1,0,1,2,40.80,36.50,28.00,62.00
3,5000004,2,2,0,0,2,1,0,1,1,42.00,42.00,33.00,51.00
4,5000005,3,3,2,0,1,0,0,2,0,36.70,44.00,20.00,46.00


In [69]:
aggregation_validation = pd.DataFrame({
    "Metric": [
        "Clean person records",
        "Total records represented after aggregation",
        "Unique collision IDs in clean person data",
        "Rows in person summary",
        "Duplicated collision IDs in person summary"
    ],
    "Value": [
        len(persons_clean),
        person_summary["person_records"].sum(),
        persons_clean["collision_id"].nunique(),
        len(person_summary),
        person_summary["collision_id"].duplicated().sum()
    ]
})

display(aggregation_validation)

,Metric,Value
0,Clean person records,8956
1,Total records represented after aggregation,8956
2,Unique collision IDs in clean person data,3191
3,Rows in person summary,3191
4,Duplicated collision IDs in person summary,0


## 5. Data Integration

A left join is used from Crashes to the aggregated Person dataset.

### Justification

- Every crash should remain in the final dataset.
- Some collisions may not have a matching usable person record.
- An inner join would incorrectly remove unmatched collisions.
- The aggregated Person dataset contains one row per collision.
- `validate="one_to_one"` is used to confirm that neither side duplicates a
  collision during integration.

In [70]:
crash_count_before_merge = len(crashes_clean)

df_integrated = crashes_clean.merge(
    person_summary,
    on="collision_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Crash rows before merge:", crash_count_before_merge)
print("Integrated rows:", len(df_integrated))

display(
    df_integrated["_merge"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Crash rows before merge: 5000
Integrated rows: 5000


,count
_merge,
both,3191
left_only,1809
right_only,0


In [71]:
matched_collisions = (
    df_integrated["_merge"] == "both"
).sum()

unmatched_collisions = (
    df_integrated["_merge"] == "left_only"
).sum()

matching_rate = (
    matched_collisions / len(df_integrated) * 100
    if len(df_integrated) > 0
    else 0
)

integration_report = pd.DataFrame({
    "Metric": [
        "Crashes before integration",
        "Rows after integration",
        "Matched collisions",
        "Unmatched collisions",
        "Matching rate (%)",
        "Duplicate collision IDs after integration"
    ],
    "Value": [
        crash_count_before_merge,
        len(df_integrated),
        matched_collisions,
        unmatched_collisions,
        round(matching_rate, 2),
        df_integrated["collision_id"].duplicated().sum()
    ]
})

display(integration_report)

,Metric,Value
0,Crashes before integration,"5,000.00"
1,Rows after integration,"5,000.00"
2,Matched collisions,"3,191.00"
3,Unmatched collisions,"1,809.00"
4,Matching rate (%),63.82
5,Duplicate collision IDs after integration,0.00


## 6. Post-Integration Cleaning

The integration introduces missing values for collisions without matching
person records.

### Post-Integration Decisions

- Missing person count fields are filled with zero.
- Missing age statistics remain missing because zero would incorrectly imply
  that the average person age was zero.
- A new field identifies whether matching Person data was available.
- Crash-level and Person-level injury totals are both retained.
- Differences between the two sources are calculated rather than silently
  replacing one source with the other.
- The merge indicator is removed after integration diagnostics are completed.

In [72]:
df_integrated["person_data_available"] = (
    df_integrated["_merge"] == "both"
)

person_count_columns = [
    "person_records",
    "unique_persons",
    "pedestrians_involved",
    "cyclists_involved",
    "occupants_involved",
    "injured_person_records",
    "killed_person_records",
    "male_persons",
    "female_persons"
]

for column in person_count_columns:
    df_integrated[column] = (
        df_integrated[column]
        .fillna(0)
        .round()
        .astype("int64")
    )

df_integrated["injury_count_difference"] = (
    df_integrated["number_of_persons_injured"]
    - df_integrated["injured_person_records"]
)

df_integrated["fatality_count_difference"] = (
    df_integrated["number_of_persons_killed"]
    - df_integrated["killed_person_records"]
)

df_integrated["person_count_category"] = pd.cut(
    df_integrated["person_records"],
    bins=[-1, 0, 1, 2, 4, np.inf],
    labels=[
        "No Person Data",
        "One Person",
        "Two People",
        "Three to Four People",
        "Five or More People"
    ]
)

df_integrated = df_integrated.drop(columns=["_merge"])

print("Post-integration cleaning completed.")
display(df_integrated.head())

Post-integration cleaning completed.


,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,cross_street_name,off_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5,crash_datetime,year,month,month_name,weekday,hour,is_weekend,time_period,total_casualties,has_injury,has_fatality,severity_category,primary_contributing_factor,primary_vehicle_type,person_records,unique_persons,pedestrians_involved,cyclists_involved,occupants_involved,injured_person_records,killed_person_records,male_persons,female_persons,average_person_age,median_person_age,minimum_person_age,maximum_person_age,person_data_available,injury_count_difference,fatality_count_difference,person_count_category
0,2013-09-14,01:04,MANHATTAN,10001,40.83,-73.91,POINT (-73.913096 40.833095),GRAND CONCOURSE,UNKNOWN,GRAND CONCOURSE,0,0,0,0,0,0,0,0,Unspecified,Unsafe Speed,Backing Unsafely,Driver Inattention/Distraction,Driver Inattention/Distraction,5000000,Station Wagon/Sport Utility Vehicle,Pick-Up Truck,Station Wagon/Sport Utility Vehicle,Sedan,Sedan,2013-09-14 01:04:00,2013,9,September,Saturday,1,True,Late Night,0,False,False,No Reported Casualty,Unspecified,Station Wagon/Sport Utility Vehicle,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,False,0,0,No Person Data
1,2022-12-13,17:30,MANHATTAN,10001,40.84,-73.92,POINT (-73.923245 40.844848),ATLANTIC AVENUE,BROADWAY,QUEENS BOULEVARD,2,0,0,0,1,0,1,0,Driver Inattention/Distraction,Unsafe Speed,Unspecified,Driver Inattention/Distraction,Failure To Yield Right-Of-Way,5000001,Motorcycle,Sedan,Sedan,Station Wagon/Sport Utility Vehicle,Sedan,2022-12-13 17:30:00,2022,12,December,Tuesday,17,False,Afternoon,2,True,False,Multiple Injuries,Driver Inattention/Distraction,Motorcycle,4,4,1,0,3,0,0,1,2,11.00,11.00,11.00,11.00,True,2,0,Three to Four People
2,2021-05-03,22:34,BROOKLYN,11201,40.66,-73.92,POINT (-73.920122 40.661667),BROADWAY,UNKNOWN,QUEENS BOULEVARD,1,0,0,0,0,0,1,0,Driver Inattention/Distraction,Traffic Control Disregarded,Unsafe Speed,Driver Inattention/Distraction,Backing Unsafely,5000002,Bus,Sedan,Motorcycle,Sedan,Bus,2021-05-03 22:34:00,2021,5,May,Monday,22,False,Night,1,True,False,Single Injury,Driver Inattention/Distraction,Bus,4,4,0,0,4,1,0,1,3,41.80,41.50,28.00,56.00,True,0,0,Three to Four People
3,2018-06-04,19:43,BROOKLYN,11201,40.61,-73.90,POINT (-73.901439 40.607136),ATLANTIC AVENUE,HYLAN BOULEVARD,BROADWAY,0,0,0,0,0,0,0,0,Failure To Yield Right-Of-Way,Driver Inattention/Distraction,Failure To Yield Right-Of-Way,Following Too Closely,Unspecified,5000003,Taxi,Taxi,Sedan,Motorcycle,Pick-Up Truck,2018-06-04 19:43:00,2018,6,June,Monday,19,False,Evening,0,False,False,No Reported Casualty,Failure To Yield Right-Of-Way,Taxi,4,4,0,1,3,1,0,1,2,40.80,36.50,28.00,62.00,True,-1,0,Three to Four People
4,2018-05-06,09:34,BROOKLYN,11201,40.71,-73.95,POINT (-73.947107 40.712138),BROADWAY,BROADWAY,UNKNOWN,0,0,0,0,0,0,0,0,Driver Inattention/Distraction,Unspecified,Driver Inattention/Distraction,Traffic Control Disregarded,Following Too Closely,5000004,Sedan,Unknown,Motorcycle,Sedan,Sedan,2018-05-06 09:34:00,2018,5,May,Sunday,9,True,Morning,0,False,False,No Reported Casualty,Driver Inattention/Distraction,Sedan,2,2,0,0,2,1,0,1,1,42.00,42.00,33.00,51.00,True,-1,0,Two People


In [73]:
casualty_comparison = pd.DataFrame({
    "Metric": [
        "Equal injury totals",
        "Different injury totals",
        "Equal fatality totals",
        "Different fatality totals"
    ],
    "Collision Count": [
        (
            df_integrated["injury_count_difference"] == 0
        ).sum(),
        (
            df_integrated["injury_count_difference"] != 0
        ).sum(),
        (
            df_integrated["fatality_count_difference"] == 0
        ).sum(),
        (
            df_integrated["fatality_count_difference"] != 0
        ).sum()
    ]
})

display(casualty_comparison)

,Metric,Collision Count
0,Equal injury totals,2909
1,Different injury totals,2091
2,Equal fatality totals,4935
3,Different fatality totals,65


### Casualty Difference Explanation

The crash table and Person table may not always report identical casualty
totals. Possible reasons include:

- Missing Person records
- Unspecified Person injury status
- Updates applied to one table before another
- Historical reporting differences
- Data-entry inconsistencies

Both measurements are retained to preserve transparency. The crash-table
casualty values remain the primary dashboard measures, while the Person values
provide additional validation and person-level analysis.

In [74]:
validation_results = {
    "One row per collision":
        df_integrated["collision_id"].is_unique,

    "No missing collision IDs":
        df_integrated["collision_id"].notna().all(),

    "No exact duplicate rows":
        not df_integrated.duplicated().any(),

    "Same crash count after left join":
        len(df_integrated) == crash_count_before_merge,

    "No negative injury counts":
        df_integrated[
            "number_of_persons_injured"
        ].ge(0).all(),

    "No negative fatality counts":
        df_integrated[
            "number_of_persons_killed"
        ].ge(0).all(),

    "Years within project range":
        df_integrated["year"].between(
            START_YEAR,
            END_YEAR
        ).all()
}

validation_table = pd.DataFrame({
    "Validation Test": validation_results.keys(),
    "Passed": validation_results.values()
})

display(validation_table)

assert all(validation_results.values()), (
    "At least one final validation test failed."
)

print("All final validation tests passed.")

,Validation Test,Passed
0,One row per collision,True
1,No missing collision IDs,True
2,No exact duplicate rows,True
3,Same crash count after left join,True
4,No negative injury counts,True
5,No negative fatality counts,True
6,Years within project range,True


All final validation tests passed.


In [75]:
integrated_missing_report = create_missing_value_report(
    df_integrated
)

display(integrated_missing_report.head(20))

,missing_count,missing_percentage,data_type
maximum_person_age,1809,36.18,float64
minimum_person_age,1809,36.18,float64
median_person_age,1809,36.18,float64
average_person_age,1809,36.18,float64
zip_code,59,1.18,string
location,35,0.70,object
longitude,35,0.70,float64
latitude,35,0.70,float64
borough,0,0.00,string
crash_time,0,0.00,string


## 7. Exploratory Visualizations and Research Insights

The following charts examine:

- Collision trends over time
- Borough casualty rates
- Time-of-day patterns
- Contributing factors
- Person-type involvement
- Geographic collision locations

In [76]:
annual_collision_summary = (
    df_integrated
    .groupby("year", as_index=False)
    .agg(
        collisions=("collision_id", "count"),
        injured=("number_of_persons_injured", "sum"),
        killed=("number_of_persons_killed", "sum")
    )
)

fig = px.line(
    annual_collision_summary,
    x="year",
    y=["collisions", "injured", "killed"],
    markers=True,
    title="Annual Collision and Casualty Trends",
    labels={
        "year": "Year",
        "value": "Count",
        "variable": "Measure"
    }
)

fig.show()

In [77]:
borough_summary = (
    df_integrated
    .groupby("borough", as_index=False)
    .agg(
        collisions=("collision_id", "count"),
        injured=("number_of_persons_injured", "sum"),
        killed=("number_of_persons_killed", "sum"),
        pedestrians=("pedestrians_involved", "sum"),
        cyclists=("cyclists_involved", "sum")
    )
)

borough_summary["casualties_per_1000_crashes"] = (
    (
        borough_summary["injured"]
        + borough_summary["killed"]
    )
    / borough_summary["collisions"]
    * 1000
).round(2)

display(
    borough_summary.sort_values(
        "casualties_per_1000_crashes",
        ascending=False
    )
)

fig = px.bar(
    borough_summary.sort_values(
        "casualties_per_1000_crashes",
        ascending=False
    ),
    x="borough",
    y="casualties_per_1000_crashes",
    hover_data=[
        "collisions",
        "injured",
        "killed",
        "pedestrians",
        "cyclists"
    ],
    title="Casualties per 1,000 Collisions by Borough",
    labels={
        "borough": "Borough",
        "casualties_per_1000_crashes":
            "Casualties per 1,000 collisions"
    }
)

fig.show()

,borough,collisions,injured,killed,pedestrians,cyclists,casualties_per_1000_crashes
0,BRONX,840,318,3,239,156,382.14
3,QUEENS,1333,450,6,391,246,342.09
1,BROOKLYN,1540,491,12,492,278,326.62
2,MANHATTAN,1049,329,8,285,168,321.26
4,STATEN ISLAND,179,56,0,71,41,312.85
5,UNKNOWN,59,18,0,15,8,305.08


In [78]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

hourly_heatmap_data = (
    df_integrated
    .pivot_table(
        index="weekday",
        columns="hour",
        values="collision_id",
        aggfunc="count",
        fill_value=0
    )
    .reindex(weekday_order)
)

fig = px.imshow(
    hourly_heatmap_data,
    aspect="auto",
    title="Collision Frequency by Weekday and Hour",
    labels={
        "x": "Hour of day",
        "y": "Weekday",
        "color": "Number of collisions"
    }
)

fig.show()

In [79]:
df_integrated["day_night_category"] = np.where(
    df_integrated["hour"].between(6, 17),
    "Daytime",
    "Nighttime"
)

day_night_summary = (
    df_integrated
    .groupby("day_night_category", as_index=False)
    .agg(
        collisions=("collision_id", "count"),
        injured=("number_of_persons_injured", "sum"),
        killed=("number_of_persons_killed", "sum")
    )
)

day_night_summary["injuries_per_1000_crashes"] = (
    day_night_summary["injured"]
    / day_night_summary["collisions"]
    * 1000
).round(2)

day_night_summary["fatalities_per_1000_crashes"] = (
    day_night_summary["killed"]
    / day_night_summary["collisions"]
    * 1000
).round(2)

display(day_night_summary)

fig = px.bar(
    day_night_summary,
    x="day_night_category",
    y=[
        "injuries_per_1000_crashes",
        "fatalities_per_1000_crashes"
    ],
    barmode="group",
    title="Daytime and Nighttime Collision Severity",
    labels={
        "day_night_category": "Time category",
        "value": "Casualties per 1,000 collisions",
        "variable": "Measure"
    }
)

fig.show()

,day_night_category,collisions,injured,killed,injuries_per_1000_crashes,fatalities_per_1000_crashes
0,Daytime,3232,1052,21,325.50,6.50
1,Nighttime,1768,610,8,345.02,4.52


In [80]:
factor_summary = (
    df_integrated[
        ~df_integrated[
            "primary_contributing_factor"
        ].isin(["Unknown", "Unspecified"])
    ]
    .groupby(
        "primary_contributing_factor",
        as_index=False
    )
    .agg(
        collisions=("collision_id", "count"),
        injured=("number_of_persons_injured", "sum"),
        killed=("number_of_persons_killed", "sum")
    )
)

factor_summary["casualties_per_1000_crashes"] = (
    (
        factor_summary["injured"]
        + factor_summary["killed"]
    )
    / factor_summary["collisions"]
    * 1000
).round(2)

top_factors = (
    factor_summary
    .sort_values("collisions", ascending=False)
    .head(15)
    .sort_values("collisions")
)

fig = px.bar(
    top_factors,
    x="collisions",
    y="primary_contributing_factor",
    orientation="h",
    hover_data=[
        "injured",
        "killed",
        "casualties_per_1000_crashes"
    ],
    title="Top Primary Contributing Factors",
    labels={
        "collisions": "Number of collisions",
        "primary_contributing_factor":
            "Primary contributing factor"
    }
)

fig.show()

In [81]:
person_type_injury_summary = (
    persons_clean
    .groupby(
        [
            "person_type_clean",
            "person_injury_clean"
        ],
        as_index=False
    )
    .size()
    .rename(columns={"size": "person_count"})
)

fig = px.bar(
    person_type_injury_summary,
    x="person_type_clean",
    y="person_count",
    color="person_injury_clean",
    barmode="group",
    title="Person Injury Status by Person Type",
    labels={
        "person_type_clean": "Person type",
        "person_count": "Number of people",
        "person_injury_clean": "Injury status"
    }
)

fig.show()

In [82]:
persons_clean["age_group"] = pd.cut(
    persons_clean["person_age"],
    bins=[-1, 12, 17, 24, 44, 64, 110],
    labels=[
        "Child",
        "Teenager",
        "Young Adult",
        "Adult",
        "Middle-Aged",
        "Senior"
    ]
)

age_injury_summary = (
    persons_clean
    .dropna(subset=["age_group"])
    .groupby(
        ["age_group", "person_injury_clean"],
        observed=True,
        as_index=False
    )
    .size()
    .rename(columns={"size": "person_count"})
)

fig = px.bar(
    age_injury_summary,
    x="age_group",
    y="person_count",
    color="person_injury_clean",
    barmode="group",
    title="Person Injury Status by Age Group",
    labels={
        "age_group": "Age group",
        "person_count": "Number of people",
        "person_injury_clean": "Injury status"
    }
)

fig.show()

In [83]:
map_data = df_integrated.dropna(
    subset=["latitude", "longitude"]
).copy()

maximum_map_points = 30_000

if len(map_data) > maximum_map_points:
    map_data = map_data.sample(
        maximum_map_points,
        random_state=RANDOM_STATE
    )

fig = px.scatter_map(
    map_data,
    lat="latitude",
    lon="longitude",
    color="severity_category",
    hover_name="borough",
    hover_data={
        "year": True,
        "crash_time": True,
        "primary_contributing_factor": True,
        "primary_vehicle_type": True,
        "number_of_persons_injured": True,
        "number_of_persons_killed": True,
        "latitude": False,
        "longitude": False
    },
    zoom=9,
    height=650,
    title="Interactive NYC Collision Map"
)

fig.update_layout(
    map_style="open-street-map",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [84]:
person_count_severity = (
    df_integrated
    .groupby(
        "person_count_category",
        observed=True,
        as_index=False
    )
    .agg(
        collisions=("collision_id", "count"),
        injured=("number_of_persons_injured", "sum"),
        killed=("number_of_persons_killed", "sum")
    )
)

person_count_severity["casualties_per_1000_crashes"] = (
    (
        person_count_severity["injured"]
        + person_count_severity["killed"]
    )
    / person_count_severity["collisions"]
    * 1000
).round(2)

display(person_count_severity)

fig = px.bar(
    person_count_severity,
    x="person_count_category",
    y="casualties_per_1000_crashes",
    title="Collision Severity by Number of People Involved",
    labels={
        "person_count_category":
            "Number of people involved",
        "casualties_per_1000_crashes":
            "Casualties per 1,000 collisions"
    }
)

fig.show()

,person_count_category,collisions,injured,killed,casualties_per_1000_crashes
0,No Person Data,1809,632,10,354.89
1,One Person,536,144,4,276.12
2,Two People,931,294,6,322.23
3,Three to Four People,1367,474,8,352.60
4,Five or More People,357,118,1,333.33


In [85]:
total_collisions = len(df_integrated)

total_injured = int(
    df_integrated["number_of_persons_injured"].sum()
)

total_killed = int(
    df_integrated["number_of_persons_killed"].sum()
)

highest_crash_borough = (
    df_integrated["borough"]
    .value_counts()
    .idxmax()
)

specified_factors = df_integrated.loc[
    ~df_integrated[
        "primary_contributing_factor"
    ].isin(["Unknown", "Unspecified"]),
    "primary_contributing_factor"
]

most_common_factor = (
    specified_factors.value_counts().idxmax()
    if not specified_factors.empty
    else "Not available"
)

print("FINAL DATASET SUMMARY")
print("-" * 60)
print(f"Project period: {START_YEAR}–{END_YEAR}")
print(f"Total collisions: {total_collisions:,}")
print(f"Total injured: {total_injured:,}")
print(f"Total killed: {total_killed:,}")
print(f"Highest collision-count borough: {highest_crash_borough}")
print(f"Most common specified factor: {most_common_factor}")
print(f"Person-data matching rate: {matching_rate:.2f}%")

FINAL DATASET SUMMARY
------------------------------------------------------------
Project period: 2012–2025
Total collisions: 5,000
Total injured: 1,662
Total killed: 29
Highest collision-count borough: BROOKLYN
Most common specified factor: Driver Inattention/Distraction
Person-data matching rate: 63.82%


## 8. Main Findings

Complete this section after running the full dataset.

### Temporal Findings

- Describe how collisions changed between 2012 and 2025.
- Identify the highest and lowest collision years.
- Explain the weekday and hourly collision patterns.
- Compare daytime and nighttime severity rates.

### Geographic Findings

- Identify the borough with the highest raw collision count.
- Compare boroughs using casualties per 1,000 collisions.
- Identify important geographic hotspots from the map.

### Contributing-Factor Findings

- Identify the most common contributing factors.
- Distinguish factors with high frequency from factors with high severity.
- Explain whether the most frequent factor is also the most dangerous factor.

### Person-Level Findings

- Compare pedestrians, cyclists and motor vehicle occupants.
- Identify age groups with the highest injury or fatality counts.
- Explain the relationship between the number of people involved and collision
  severity.

### Data-Quality Findings

- Report the integration matching rate.
- Explain differences between crash-table and Person-table casualty totals.
- Discuss the effect of cleaning on duplicates, missing values and invalid data.

## 9. Preparing Data for the Interactive Website

The final integrated data is exported to Parquet because Parquet:

- Uses less storage than CSV
- Preserves data types
- Loads more quickly
- Is suitable for Dash and Plotly applications

A smaller CSV sample is also exported for inspection and testing.

In [86]:
dashboard_columns = [
    "collision_id",
    "crash_date",
    "crash_time",
    "crash_datetime",
    "year",
    "month",
    "month_name",
    "weekday",
    "hour",
    "time_period",
    "is_weekend",
    "day_night_category",
    "borough",
    "zip_code",
    "latitude",
    "longitude",
    "on_street_name",
    "cross_street_name",
    "number_of_persons_injured",
    "number_of_persons_killed",
    "total_casualties",
    "has_injury",
    "has_fatality",
    "severity_category",
    "primary_contributing_factor",
    "primary_vehicle_type",
    "person_data_available",
    "person_records",
    "unique_persons",
    "person_count_category",
    "pedestrians_involved",
    "cyclists_involved",
    "occupants_involved",
    "injured_person_records",
    "killed_person_records",
    "male_persons",
    "female_persons",
    "average_person_age",
    "median_person_age",
    "injury_count_difference",
    "fatality_count_difference"
]

dashboard_columns = [
    column for column in dashboard_columns
    if column in df_integrated.columns
]

dashboard_data = df_integrated[dashboard_columns].copy()

print("Dashboard dataset shape:", dashboard_data.shape)
display(dashboard_data.head())

Dashboard dataset shape: (5000, 41)


,collision_id,crash_date,crash_time,crash_datetime,year,month,month_name,weekday,hour,time_period,is_weekend,day_night_category,borough,zip_code,latitude,longitude,on_street_name,cross_street_name,number_of_persons_injured,number_of_persons_killed,total_casualties,has_injury,has_fatality,severity_category,primary_contributing_factor,primary_vehicle_type,person_data_available,person_records,unique_persons,person_count_category,pedestrians_involved,cyclists_involved,occupants_involved,injured_person_records,killed_person_records,male_persons,female_persons,average_person_age,median_person_age,injury_count_difference,fatality_count_difference
0,5000000,2013-09-14,01:04,2013-09-14 01:04:00,2013,9,September,Saturday,1,Late Night,True,Nighttime,MANHATTAN,10001,40.83,-73.91,GRAND CONCOURSE,UNKNOWN,0,0,0,False,False,No Reported Casualty,Unspecified,Station Wagon/Sport Utility Vehicle,False,0,0,No Person Data,0,0,0,0,0,0,0,NaN,NaN,0,0
1,5000001,2022-12-13,17:30,2022-12-13 17:30:00,2022,12,December,Tuesday,17,Afternoon,False,Daytime,MANHATTAN,10001,40.84,-73.92,ATLANTIC AVENUE,BROADWAY,2,0,2,True,False,Multiple Injuries,Driver Inattention/Distraction,Motorcycle,True,4,4,Three to Four People,1,0,3,0,0,1,2,11.00,11.00,2,0
2,5000002,2021-05-03,22:34,2021-05-03 22:34:00,2021,5,May,Monday,22,Night,False,Nighttime,BROOKLYN,11201,40.66,-73.92,BROADWAY,UNKNOWN,1,0,1,True,False,Single Injury,Driver Inattention/Distraction,Bus,True,4,4,Three to Four People,0,0,4,1,0,1,3,41.80,41.50,0,0
3,5000003,2018-06-04,19:43,2018-06-04 19:43:00,2018,6,June,Monday,19,Evening,False,Nighttime,BROOKLYN,11201,40.61,-73.90,ATLANTIC AVENUE,HYLAN BOULEVARD,0,0,0,False,False,No Reported Casualty,Failure To Yield Right-Of-Way,Taxi,True,4,4,Three to Four People,0,1,3,1,0,1,2,40.80,36.50,-1,0
4,5000004,2018-05-06,09:34,2018-05-06 09:34:00,2018,5,May,Sunday,9,Morning,True,Daytime,BROOKLYN,11201,40.71,-73.95,BROADWAY,BROADWAY,0,0,0,False,False,No Reported Casualty,Driver Inattention/Distraction,Sedan,True,2,2,Two People,0,0,2,1,0,1,1,42.00,42.00,-1,0


In [87]:
output_directory = Path("data/processed")
output_directory.mkdir(parents=True, exist_ok=True)

parquet_output_path = (
    output_directory
    / "integrated_collisions.parquet"
)

csv_sample_output_path = (
    output_directory
    / "integrated_collisions_sample.csv"
)

schema_output_path = (
    output_directory
    / "integrated_collisions_schema.csv"
)

dashboard_data.to_parquet(
    parquet_output_path,
    index=False
)

sample_size = min(100_000, len(dashboard_data))

if sample_size > 0:
    dashboard_data.sample(
        n=sample_size,
        random_state=RANDOM_STATE
    ).to_csv(
        csv_sample_output_path,
        index=False
    )
else:
    dashboard_data.to_csv(
        csv_sample_output_path,
        index=False
    )

schema_table = pd.DataFrame({
    "column_name": dashboard_data.columns,
    "data_type": dashboard_data.dtypes.astype(str).values,
    "missing_count": dashboard_data.isna().sum().values,
    "missing_percentage": (
        dashboard_data.isna().mean().values * 100
    ).round(2)
})

schema_table.to_csv(
    schema_output_path,
    index=False
)

print("Files exported successfully:")
print(parquet_output_path)
print(csv_sample_output_path)
print(schema_output_path)

Files exported successfully:
data/processed/integrated_collisions.parquet
data/processed/integrated_collisions_sample.csv
data/processed/integrated_collisions_schema.csv


In [88]:
dashboard_test = pd.read_parquet(
    "data/processed/integrated_collisions.parquet"
)

print("Exported file loaded successfully.")
print("Loaded dimensions:", dashboard_test.shape)

display(dashboard_test.head())

Exported file loaded successfully.
Loaded dimensions: (5000, 41)


,collision_id,crash_date,crash_time,crash_datetime,year,month,month_name,weekday,hour,time_period,is_weekend,day_night_category,borough,zip_code,latitude,longitude,on_street_name,cross_street_name,number_of_persons_injured,number_of_persons_killed,total_casualties,has_injury,has_fatality,severity_category,primary_contributing_factor,primary_vehicle_type,person_data_available,person_records,unique_persons,person_count_category,pedestrians_involved,cyclists_involved,occupants_involved,injured_person_records,killed_person_records,male_persons,female_persons,average_person_age,median_person_age,injury_count_difference,fatality_count_difference
0,5000000,2013-09-14,01:04,2013-09-14 01:04:00,2013,9,September,Saturday,1,Late Night,True,Nighttime,MANHATTAN,10001,40.83,-73.91,GRAND CONCOURSE,UNKNOWN,0,0,0,False,False,No Reported Casualty,Unspecified,Station Wagon/Sport Utility Vehicle,False,0,0,No Person Data,0,0,0,0,0,0,0,NaN,NaN,0,0
1,5000001,2022-12-13,17:30,2022-12-13 17:30:00,2022,12,December,Tuesday,17,Afternoon,False,Daytime,MANHATTAN,10001,40.84,-73.92,ATLANTIC AVENUE,BROADWAY,2,0,2,True,False,Multiple Injuries,Driver Inattention/Distraction,Motorcycle,True,4,4,Three to Four People,1,0,3,0,0,1,2,11.00,11.00,2,0
2,5000002,2021-05-03,22:34,2021-05-03 22:34:00,2021,5,May,Monday,22,Night,False,Nighttime,BROOKLYN,11201,40.66,-73.92,BROADWAY,UNKNOWN,1,0,1,True,False,Single Injury,Driver Inattention/Distraction,Bus,True,4,4,Three to Four People,0,0,4,1,0,1,3,41.80,41.50,0,0
3,5000003,2018-06-04,19:43,2018-06-04 19:43:00,2018,6,June,Monday,19,Evening,False,Nighttime,BROOKLYN,11201,40.61,-73.90,ATLANTIC AVENUE,HYLAN BOULEVARD,0,0,0,False,False,No Reported Casualty,Failure To Yield Right-Of-Way,Taxi,True,4,4,Three to Four People,0,1,3,1,0,1,2,40.80,36.50,-1,0
4,5000004,2018-05-06,09:34,2018-05-06 09:34:00,2018,5,May,Sunday,9,Morning,True,Daytime,BROOKLYN,11201,40.71,-73.95,BROADWAY,BROADWAY,0,0,0,False,False,No Reported Casualty,Driver Inattention/Distraction,Sedan,True,2,2,Two People,0,0,2,1,0,1,1,42.00,42.00,-1,0


## 10. Conclusion

This notebook completed the main data engineering stages required for
Milestone 1:

1. The NYC Crashes and Person datasets were loaded and explored.
2. Missing values, duplicates, invalid values and inconsistent categories were
   investigated.
3. Pre-integration cleaning was applied independently to both datasets.
4. Person records were aggregated to preserve one row per collision.
5. The datasets were integrated using a validated left join.
6. New missing values and data-type issues were resolved after integration.
7. The final dataset was validated using explicit data-quality tests.
8. Interactive visualizations were created to investigate temporal,
   geographic, severity and person-level patterns.
9. The final dashboard-ready data was exported in Parquet format.

The exported dataset can now be used by the interactive Dash website with
dropdown filters, natural-language search and the Generate Report button.